## Calorie Expenditure Prediction

This project predicts the number of calories burned during a workout from simple body and exercise measurements: sex, age, height, weight, workout duration, heart rate, and body temperature.

## Approach
1. Load the data and explore the target and features
2. Encode the categorical feature (Sex)
3. Log-transform the target to match the RMSLE metric
4. Train a baseline regression model and evaluate it on a validation split
5. Generate predictions and create the submission file
6. Submit to Kaggle and record the score

In [1]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token and press Enter: ")

!pip install -q -U kaggle
!kaggle competitions download -c playground-series-s5e5
!unzip -oq playground-series-s5e5.zip

Paste your Kaggle API token and press Enter: ··········
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 7.3 MB/s eta 0:00:00
100% 12.5M/12.5M [00:00<00:00, 117MB/s]



In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [3]:
train.shape, test.shape

((750000, 9), (250000, 8))

In [4]:
train.sample(1)

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
188043,188043,female,32,153.0,52.0,20.0,103.0,40.3,121.0


In [5]:
test.sample(1)

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
8962,758962,male,44,174.0,81.0,26.0,112.0,40.8


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   id          750000 non-null  int64  
 1   Sex         750000 non-null  object 
 2   Age         750000 non-null  int64  
 3   Height      750000 non-null  float64
 4   Weight      750000 non-null  float64
 5   Duration    750000 non-null  float64
 6   Heart_Rate  750000 non-null  float64
 7   Body_Temp   750000 non-null  float64
 8   Calories    750000 non-null  float64
dtypes: float64(6), int64(2), object(1)
memory usage: 51.5+ MB


In [7]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   id          250000 non-null  int64  
 1   Sex         250000 non-null  object 
 2   Age         250000 non-null  int64  
 3   Height      250000 non-null  float64
 4   Weight      250000 non-null  float64
 5   Duration    250000 non-null  float64
 6   Heart_Rate  250000 non-null  float64
 7   Body_Temp   250000 non-null  float64
dtypes: float64(5), int64(2), object(1)
memory usage: 15.3+ MB


In [8]:
train.columns.tolist()

['id',
 'Sex',
 'Age',
 'Height',
 'Weight',
 'Duration',
 'Heart_Rate',
 'Body_Temp',
 'Calories']

In [9]:
train["Calories"].describe()

,Calories
count,750000.000000
mean,88.282781
std,62.395349
min,1.000000
25%,34.000000
50%,77.000000
75%,136.000000
max,314.000000


In [10]:
train.columns.tolist()

['id',
 'Sex',
 'Age',
 'Height',
 'Weight',
 'Duration',
 'Heart_Rate',
 'Body_Temp',
 'Calories']

In [11]:
test_ids = test["id"]
train = train.drop(columns=["id"])
test = test.drop(columns=["id"])

In [12]:
train = pd.get_dummies(train, columns=["Sex"])

In [13]:
test = pd.get_dummies(test, columns=["Sex"])

In [14]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   Age         750000 non-null  int64  
 1   Height      750000 non-null  float64
 2   Weight      750000 non-null  float64
 3   Duration    750000 non-null  float64
 4   Heart_Rate  750000 non-null  float64
 5   Body_Temp   750000 non-null  float64
 6   Calories    750000 non-null  float64
 7   Sex_female  750000 non-null  bool   
 8   Sex_male    750000 non-null  bool   
dtypes: bool(2), float64(6), int64(1)
memory usage: 41.5 MB


In [15]:
train, test = train.align(test, join="left", axis=1, fill_value=0)

In [16]:
print(train.shape, test.shape)

(750000, 9) (250000, 9)


In [18]:
train.columns.tolist()

['Age',
 'Height',
 'Weight',
 'Duration',
 'Heart_Rate',
 'Body_Temp',
 'Calories',
 'Sex_female',
 'Sex_male']

In [19]:
x = train.drop(columns=["Calories"])
y = np.log1p(train["Calories"])

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, max_depth=12, min_samples_leaf=10, random_state=42, n_jobs=-1)
model.fit(x_train, y_train)

pred = model.predict(x_val)
rmsle = mean_squared_error(y_val, pred) ** 0.5
print("Validation RMSLE:", round(rmsle, 4))

Validation RMSLE: 0.0644


In [21]:
model.fit(x, y)

x_test = test.drop(columns=["Calories"])
pred_test = np.expm1(model.predict(x_test))

pd.DataFrame({"id": test_ids, "Calories": pred_test}).to_csv("submission.csv", index=False)

In [22]:
!kaggle competitions submit -c playground-series-s5e5 -f submission.csv -m "RandomForest baseline"

100% 6.06M/6.06M [00:00<00:00, 14.2MB/s]
99 submissions remaining today.
Successfully submitted to Predict Calorie Expenditure

In [23]:
model = RandomForestRegressor(n_estimators=60, max_depth=10, min_samples_leaf=20, random_state=42, n_jobs=-1)
model.fit(x, y)

import pickle, os
pickle.dump(model, open("calorie_model.pkl", "wb"))
print("size MB:", round(os.path.getsize("calorie_model.pkl") / 1e6, 1))

size MB: 8.7
